# B-spline examples with K > 0

Pick a name in the catalogue **`docs/bspline_examples.html`** (open it in a browser; rebuild it with
`python scripts/build_example_catalog.py`), paste it as `NAME` below and run the cells.
Any `loft_<seed>` works, not only the seeds shown in the catalogue.

1. run the setup cell once,
2. **Show** — fits `NAME` with its catalogue settings and shows it coloured by K,
3. **Change parameters** — optional, same parameters as the catalogue table,
4. **Save** — optional, writes `data/bsplines/<NAME>.json` for `lconjugacy_lab.ipynb`.

In [ ]:
# Setup (run once). Single-threaded BLAS: must run before numpy is imported.
import os, sys
for _var in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_var, "1")

%load_ext autoreload
%autoreload 2

import numpy as np

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import kayviz as kv
import Lconjugacy.surface_examples as ex

kv.init()

def show(bsp, label, n=100, net=False):
    """Show the B-spline coloured by Gaussian curvature (and its sign)."""
    V, F, K = ex.mesh(bsp, n, n)
    kv.clear()
    kv.register_surface_mesh(label, V, F, color=(0.6, 0.6, 0.6))
    kv.add_scalar_quantity(label, "Gaussian curvature", K)
    kv.add_scalar_quantity(label, "sign K", np.sign(K))
    if net:
        C, E = ex.control_net(bsp)
        kv.register_curve_network(label + " control net", C, E, color=(1, 0, 0), radius=0.002)

def report(bsp, label):
    r = ex.curvature_report(bsp)
    m, k = np.asarray(bsp.controlpoints).shape[:2]
    print(f"{label}: {m}x{k} control points, degree {bsp.order(0) - 1}, "
          f"K in [{r['K_min']:.3g}, {r['K_max']:.3g}], K_max/K_min = {r['K_ratio']:.3g}, "
          f"share K <= 0: {r['frac_K_nonpositive']:.1%}")

## Show an example

In [ ]:
NAME = "loft_vault"          # any name from docs/bspline_examples.html, or loft_<seed>

bsp = ex.fit(NAME)
report(bsp, NAME)
show(bsp, NAME, net=False)

## Change parameters

`params` holds the values of the catalogue table. Edit them and run the next cell.
For tubes and shells (`spiral_shell`, `conch_shell`, `nautilus_shell`, ...) only the outer band is
K > 0: after changing their parameters, set `params["u_range"] = ex.k_positive_u_range(NAME, u_search=(-np.pi, np.pi), **params)`.
Check that "share K ≤ 0" stays 0 %.

In [ ]:
params = ex.get_params(NAME)
params

In [ ]:
params.update(
    # seed=7,
    # bulge=2.5,
)
bsp = ex.fit(NAME, **params)
report(bsp, NAME)
show(bsp, NAME)

## Save

Writes the current `bsp` to `data/bsplines/<NAME>.json`; in `lconjugacy_lab.ipynb` use
`SURFACE = NAME`. (`read_bspline_json` rescales the control points into a box of size 2.)

In [ ]:
ex.save(bsp, NAME)